# Issue #61: mvSuSiE with Large Number of Traits (R=128)

Field test for https://github.com/stephenslab/mvsusieR/issues/61

When N < R (more traits than samples), `cov(Y)` is singular, causing
the default residual variance initialization to fail. The fix: fall
back to flash-based covariance estimation when `cov(Y)` is not PD.

In [1]:
library(mvsusieR)
library(fsusieR)
library(susieR)
library(ggplot2)
library(cowplot)

Loading required package: mashr

Loading required package: ashr

Loading required package: susieR


Attaching package: ‘fsusieR’


The following object is masked from ‘package:ashr’:

    get_pi0




## Simulate data from Issue #61

In [2]:
set.seed(1)
genotypes <- N3finemapping$X[sample(1:nrow(N3finemapping$X), size = 100), ]

s <- 7  # 2^7 = 128 traits
L <- 3
lf <- list()
for (l in 1:L) {
  lf[[l]] <- simu_IBSS_per_level(lev_res = s)$sim_func
}

G <- genotypes
zero_var <- which(apply(G, 2, var) == 0)
if (length(zero_var) > 0) G <- G[, -zero_var]

true_pos <- sample(1:ncol(G), L)
Y <- matrix(0, ncol = 2^s, nrow = 100)
for (i in 1:100) {
  for (l in 1:L) {
    Y[i, ] <- Y[i, ] + lf[[l]] * G[i, true_pos[[l]]]
  }
}
Y <- Y + matrix(rnorm((2^s) * 100, sd = sd(c(Y))), nrow = 100)

cat("X dims:", dim(G), "\n")
cat("Y dims:", dim(Y), "\n")
cat("N < R:", nrow(G) < ncol(Y), "\n")
cat("True causal positions:", true_pos, "\n")

X dims: 100 986 
Y dims: 100 128 
N < R: TRUE 
True causal positions: 933 823 842 


## Verify cov(Y) is not PD

In [3]:
V_cov <- cov(Y)
eig_vals <- eigen(V_cov, symmetric = TRUE, only.values = TRUE)$values
cat("Rank of cov(Y):", sum(eig_vals > 1e-10), "/", ncol(Y), "\n")
cat("Min eigenvalue:", min(eig_vals), "\n")
cat("Number of negative eigenvalues:", sum(eig_vals < 0), "\n")
tryCatch({
  chol(V_cov)
  cat("Cholesky succeeded (PD)\n")
}, error = function(e) {
  cat("Cholesky failed (not PD):", e$message, "\n")
})

Rank of cov(Y): 99 / 128 
Min eigenvalue: -7.022983e-15 
Number of negative eigenvalues: 15 
Cholesky failed (not PD): the leading minor of order 100 is not positive 


## Fit mvSuSiE with defaults

In [5]:
prior <- create_mixture_prior(R = ncol(Y))

In [4]:
t0 <- proc.time()
m1 <- mvsusie(X = G, Y = Y, prior_variance = prior, L = L)
t1 <- proc.time()
walltime <- (t1 - t0)[3]
cat("Wall time:", round(walltime, 1), "seconds\n")
cat("Converged:", m1$convergence$converged, "\n")
cat("Iterations:", m1$niter, "\n")
cat("Number of CSs:", length(m1$sets$cs), "\n")

mvsusie: N=100, J=986, R=128, L=3 [mem: 0.22 GB]



Warning message:
“cov(Y) is not positive definite (N < R or collinear traits); adding ridge to enforce positive definiteness.”


Residual variance set, common_cov=TRUE [mem: 0.22 GB]



Prior: K=133 mixture components [mem: 0.22 GB]



Eigendecomposition cache: K=133, common_cov=TRUE [mem: 0.25 GB]



...

iter   3: ELBO=542634.1618, delta=0.00e+00 -- converged [mem: 0.26 GB]



Wall time: 473.7 seconds


Converged: TRUE 


Iterations: 3 


Number of CSs: 0 


## Save results

In [5]:
results <- list(
  fit = m1,
  true_pos = true_pos,
  walltime = walltime,
  G_dims = dim(G),
  Y_dims = dim(Y),
  L = L
)
saveRDS(results, "issue61_results.rds")
cat("Results saved to issue61_results.rds\n")

Results saved to issue61_results.rds


## Fit mvSuSiE without estimating residual variance

To reproduce @pcarbo's analysis in Issue 61.

In [7]:
t0 <- proc.time()
m1 <- mvsusie(X = G, Y = Y, prior_variance = prior, L = L, estimate_residual_variance = FALSE)
t1 <- proc.time()
walltime <- (t1 - t0)[3]
cat("Wall time:", round(walltime, 1), "seconds\n")
cat("Converged:", m1$convergence$converged, "\n")
cat("Iterations:", m1$niter, "\n")
cat("Number of CSs:", length(m1$sets$cs), "\n")

mvsusie: N=100, J=986, R=128, L=3 [mem: 0.22 GB]

Warning message:
“cov(Y) is not positive definite (N < R or collinear traits); adding ridge to enforce positive definiteness.”
Residual variance set, common_cov=TRUE [mem: 0.22 GB]

Prior: K=133 mixture components [mem: 0.22 GB]

Eigendecomposition cache: K=133, common_cov=TRUE [mem: 0.25 GB]

Model initialized: J=986, R=128, L=3, K=133 [mem: 0.25 GB]

iter   2: ELBO=11104.5142, delta=0.00e+00 -- converged [mem: 0.26 GB]



Wall time: 58.7 seconds
Converged: TRUE 
Iterations: 2 
Number of CSs: 0 
